In [25]:
import sys
sys.path.append("..")

import torch
import torchmetrics
from lightning import Trainer, seed_everything
from src import LightningDataset, Module
from src.constants import DEFAULT_SEED
from src.datasets import CustomDataset
from src.transforms import LineGraph

In [26]:
CKPT = "../lightning_logs/version_45927073/checkpoints/epoch=101-step=131378.ckpt"
BATCH_SIZE = 1

In [27]:
_ = seed_everything(DEFAULT_SEED, verbose=False)

In [28]:
ckpt = torch.load(CKPT, map_location="cpu")
k = ckpt["datamodule_hyper_parameters"]["k"]

/tmp/ipykernel_105684/3996809755.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT, map_location="cpu")


In [29]:
from pprint import pprint

ckpt_params = ckpt["hyper_parameters"] | ckpt["datamodule_hyper_parameters"]
pprint(ckpt_params)

{'batch_size': 256,
 'compile': True,
 'dataset': None,
 'dataset_name': 'csg',
 'force_reload': False,
 'k': 14,
 'lengths': (0.7, 0.2, 0.1),
 'lr': 0.0027542287033381664,
 'max_iters': 386700,
 'model_kwargs': {'angle_expansion_units': 128,
                  'classification_layers': 3,
                  'classification_units': 512,
                  'dropout': 0.1,
                  'edge_expansion_units': 256,
                  'n_bond_conv': 6,
                  'num_radial': 120},
 'model_name': 'cegann',
 'num_classes': 156,
 'num_workers': 8,
 'pre_filters': None,
 'pre_transforms': LineGraph(),
 'pred_dataset': None,
 'transforms': RandomPerturbation(stddev=0.1),
 'use_imbalance_sampler': True,
 'warmup': 100}


In [30]:
datamodule = LightningDataset(
    pred_dataset=CustomDataset(root="../data/test", pre_transform=LineGraph(), k=k),
    num_workers=4,
    batch_size=BATCH_SIZE,
    k=ckpt_params["k"],
)

In [31]:
metrics = torchmetrics.MetricCollection(
    {
        "f1": torchmetrics.F1Score(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "auroc": torchmetrics.AUROC(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "acc": torchmetrics.Accuracy(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "confmat": torchmetrics.ConfusionMatrix(
            task="multiclass", num_classes=ckpt_params["num_classes"]
        ),
    }
)

In [32]:
model = Module.load_from_checkpoint(checkpoint_path=CKPT, metrics=metrics)

In [33]:
trainer = Trainer(
    precision="16-mixed" if torch.cuda.is_available() else 32,
    deterministic=False,
    enable_progress_bar=True,
)

Using 16bit Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [34]:
predictions = trainer.predict(model=model, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0:   0%|          | 0/1 [16:51:37<?, ?it/s]


BackendCompilerFailed: backend='inductor' raised:
AssertionError: increase TRITON_MAX_BLOCK['X'] to 4096

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True
